This is a notebook to visualize different properties of the dataset as we go about curating. Here, we make datasets based on [Gene Ontology](https://geneontology.org/) annotations that provide information on protein functions. These can be thought of as complementary information schemes. There are three main branches:
- **GO cellular compartment** CC serves to capture the cellular location where a molecular function takes place
- **GO molecular function** MFs represent molecular-level activities performed by gene products, such as “catalysis” or “transcription regulator activity”. MFs correspond to activities that can be performed by individual gene products (i.e. a protein or RNA), but some activities are performed by molecular complexes composed of multiple gene products, when the activity cannot be ascribed to a single gene product of the complex.
- **GO biological process** BPs are the larger processes or ‘biological programs’ accomplished by the concerted action of multiple molecular activities.

In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import polars as pl
from project.utils.strs import SEED, data_dir
from project.utils.splitting import hierarchical_clustered_split
from project.utils.functions import one_hot_polars_column, check_correlation_plotly

In [ ]:
subset_dir = data_dir / 'processed_subsets'
go_dir = subset_dir / 'go'
go_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Read annotated dataset
annotated_data = 'uniprotkb_AND_model_organism_9606_2025_09_09_annotated.parquet.gz'
df = pl.read_parquet(data_dir / annotated_data)

### Processing: one-hot transformation, thresholding and splitting

In [ ]:
num_splits = 3

# One-hot the gene ontology columns
go_columns = ['Gene Ontology (biological process)',
 'Gene Ontology (cellular component)',
 'Gene Ontology (molecular function)',]

# Thresholds - must be separate clusters in this mmseqs category to be put into test set
val_threshold_value = 50 # 40% identity minimum
test_threshold_value = 20 #0.20

val_threshold = f'mmseqs_0.{val_threshold_value}'
test_threshold = f'mmseqs_0.{test_threshold_value}'
split_count_threshold = 6 # must have at least n samples in the val set and the test set - we shouldn't have the same proteins, but we want proteins of the same pathway / compartment / function
mmseqs_cols = [col for col in df.columns if ('mmseqs' in col)]
cols_to_keep = ['id', 'sequence'] + mmseqs_cols + ['split']

dfs_onehot = {}
for col_name in go_columns:
    for i in range(num_splits):
        if i !=0:
            appendix = '_split' + str(i)
        else:
            appendix = ''

        # Convert all terms higher than a given number of counts to one-hot, and drop rows that are all negative
        onehot_df = one_hot_polars_column(col_name, df, cols_to_keep[:-1], delimiter=';', occurrence_threshold=1, only_get_positive=True)
        # Apply train-val-test splits
        onehot_df = pl.concat(hierarchical_clustered_split(onehot_df, 
                    val_threshold = val_threshold,
                    test_threshold = test_threshold,
                    val_ratio = 0.15,
                    test_ratio = 0.15,
                    seed = SEED+i))

        # Calculate sums and identify valid columns using Polars expressions
        
        # Calculate sums for ALL one-hot columns across the two splits
        col_sums = onehot_df.group_by(pl.col('split')) \
            .agg(pl.exclude(cols_to_keep).sum())
            
        # Get the row corresponding to the test set sums
        test_col_sums = col_sums.filter(pl.col('split') == 'test').drop('split')
        # Get the row corresponding to the val set sums
        val_col_sums = col_sums.filter(pl.col('split') == 'val').drop('split')
        
        # Convert sum Polars DataFrames to dictionaries (or just the list of values)
        # to perform the comparison.
        test_counts = test_col_sums.to_dict(as_series=False)
        val_counts = val_col_sums.to_dict(as_series=False)
        
        # Get the list of all one-hot column names
        all_onehot_cols = test_col_sums.columns
        
        # Determine valid columns based on the split_count_threshold
        valid_cols = [
            col for col in all_onehot_cols
            # Check if the list (containing the single sum) is greater than the threshold
            if (test_counts[col][0] > split_count_threshold) and 
            (val_counts[col][0] > split_count_threshold)
        ]
        
        print(f"  Total GO terms: {len(all_onehot_cols)}. Keeping {len(valid_cols)} terms.")
        
        rows_to_keep_mask = pl.sum_horizontal(valid_cols) > 0
        filtered_onehot_df = onehot_df.filter(rows_to_keep_mask)

        # 4. Filter the DataFrame
        dfs_onehot[col_name] = filtered_onehot_df.select(cols_to_keep + valid_cols)

        print(f"  Rows before filtering: {len(onehot_df)}. Rows after filtering: {len(filtered_onehot_df)}.")

        # Make a version with only samples less than length 512
        print(filtered_onehot_df['split'].value_counts(normalize=True)) # Make sure the composition is ~ the same before and after subsetting
        df_512 = filtered_onehot_df.filter(pl.col('sequence').str.len_chars() <= 512)
        print('after subsetting')
        print(df_512['split'].value_counts(normalize=True))
        df_512.write_parquet(go_dir / f"h_sapiens_proteome_go_{col_name.split('(')[1].split(')')[0].replace(' ', '_')}_clustersplit_{test_threshold_value}_{val_threshold_value}_512_cutoff{appendix}.parquet.gz")

        filtered_onehot_df.write_parquet(go_dir / f"h_sapiens_proteome_go_{col_name.split('(')[1].split(')')[0].replace(' ', '_')}_clustersplit_{test_threshold_value}_{val_threshold_value}{appendix}.parquet.gz")

In [ ]:
dfs_onehot

Visual confirmation that we have a good level of abstraction in GO terms - nothing too specific that would be unfairly hard for models

### Checking what terms are correlated with each other

In [ ]:
check_correlation_plotly(dfs_onehot[go_columns[0]], color_map='viridis', height=1500, width = 1500)

&rarr; the biological process terms are largely uncorrelated with each other

In [ ]:
check_correlation_plotly(dfs_onehot[go_columns[1]], color_map='viridis', height=1500, width = 1500)

&rarr; the cell component terms are largely correlated by compartment; they are terms at different levels of granularity. There's generally a higher correlation here than with the other datasets.

In [ ]:
check_correlation_plotly(dfs_onehot[go_columns[2]], color_map='viridis', height=1500, width = 1500)

&rarr; the molecular function terms are largely uncorrelated with each other, and those that are are grouped at different levels of biological granularity (e.g. there is an RNA pol II group within a DNA-binding group)